---
title: "Chapter -- Support Vector Machines"
jupyter: python3

execute: 
  enabled: true
---

## Introduction

Support Vector Machines (SVMs) are supervised learning models that can be used for classification, regression, and outlier detection. Their cenztral idea is geometric: instead of merely finding a decision boundary that separates classes, an SVM seeks a boundary with the **largest possible margin** between the classes.

SVMs are especially effective for small- and medium-sized datasets with complex decision boundaries. They can model both linear and nonlinear relationships, but they are sensitive to feature scaling and their performance depends strongly on the choice of hyperparameters.

::: {.callout-note}
## Learning objectives

After completing this chapter, you should be able to:

- explain the concepts of separating hyperplane, margin, and support vectors;
- distinguish hard-margin and soft-margin classification;
- interpret the hyperparameter `C`;
- explain why feature scaling is essential for SVMs;
- use linear, polynomial, and radial basis function kernels;
- interpret the role of `gamma` in an RBF kernel;
- fit SVM classifiers and regressors with Scikit-Learn;
- diagnose underfitting and overfitting in an SVM model.
:::

In [ ]:
#| label: imports
#| include: false

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, make_moons
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.svm import LinearSVC, SVC, LinearSVR, SVR

## Linear SVM classification

Consider a binary classification problem with a feature vector

$$
\mathbf{x} =
\begin{bmatrix}
x_1 \\
x_2 \\
\vdots \\
x_p
\end{bmatrix}.
$$

A linear classifier uses a decision function of the form

$$
f(\mathbf{x}) = \mathbf{w}^{\mathsf T}\mathbf{x} + b,
$$

where $\mathbf{w}$ contains the model coefficients and $b$ is the intercept.

The decision boundary is the hyperplane

$$
\mathbf{w}^{\mathsf T}\mathbf{x} + b = 0.
$$

For binary labels $y \in \{-1,+1\}$, the predicted class is determined by the sign of the decision function:

$$
\widehat{y} =
\begin{cases}
+1, & f(\mathbf{x}) \geq 0,\\
-1, & f(\mathbf{x}) < 0.
\end{cases}
$$

In two dimensions, the decision boundary is a line. In three dimensions, it is a plane. In higher-dimensional spaces, it is called a hyperplane.

### Margin and support vectors

Many different hyperplanes may separate a linearly separable dataset. An SVM selects the one that maximizes the distance between the decision boundary and the closest observations from each class.

The two margin boundaries are

$$
\mathbf{w}^{\mathsf T}\mathbf{x} + b = 1
$$

and

$$
\mathbf{w}^{\mathsf T}\mathbf{x} + b = -1.
$$

The distance between them is

$$
\frac{2}{\lVert\mathbf{w}\rVert}.
$$

Therefore, maximizing the margin is equivalent to minimizing $\lVert\mathbf{w}\rVert$.

The observations that lie on or inside the margin are called **support vectors**. These observations determine the position of the decision boundary. Observations located far from the margin generally do not affect the fitted hyperplane.

### A maximum-margin classifier

The following example uses two Iris features to illustrate a linear SVM. The figure is generated entirely with Python.

In [ ]:
#| label: fig-linear-svm
#| fig-cap: Linear SVM classification and its maximum-margin decision boundary.
#| code-fold: true
#| code-summary: Show code
iris = load_iris(as_frame=True)

X = iris.data.loc[
    iris.target.isin([0, 1]),
    ["petal length (cm)", "petal width (cm)"]
].to_numpy()

y = iris.target[iris.target.isin([0, 1])].to_numpy()

linear_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear", C=1_000)
)

linear_svm.fit(X, y)

fig, ax = plt.subplots(figsize=(7, 5))

DecisionBoundaryDisplay.from_estimator(
    linear_svm,
    X,
    response_method="decision_function",
    plot_method="contour",
    levels=[-1, 0, 1],
    linestyles=["--", "-", "--"],
    ax=ax
)

ax.scatter(
    X[y == 0, 0],
    X[y == 0, 1],
    marker="o",
    label="Iris setosa"
)

ax.scatter(
    X[y == 1, 0],
    X[y == 1, 1],
    marker="s",
    label="Iris versicolor"
)

support_vectors_scaled = linear_svm.named_steps["svc"].support_vectors_
support_vectors = linear_svm.named_steps["standardscaler"].inverse_transform(
    support_vectors_scaled
)

ax.scatter(
    support_vectors[:, 0],
    support_vectors[:, 1],
    s=180,
    facecolors="none",
    linewidths=1.5,
    label="Support vectors"
)

ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
ax.legend()
plt.show()

The solid line represents the decision boundary. The dashed lines represent the margins, and the circled observations are the support vectors.

## Why feature scaling matters

SVMs are sensitive to differences in feature scales because the margin depends on geometric distances. A feature measured over a wide numerical range may dominate another feature whose values occupy a much narrower range.

The next example compares the decision boundary before and after standardization.

In [ ]:
#| label: fig-svm-scaling
#| fig-cap: Effect of feature scaling on a linear SVM.
#| code-fold: true
#| code-summary: Show code

X_scale = np.array([
    [1.0, 50],
    [5.0, 20],
    [3.0, 80],
    [5.0, 60]
])

y_scale = np.array([0, 0, 1, 1])

unscaled_model = SVC(kernel="linear", C=100)
unscaled_model.fit(X_scale, y_scale)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_scale)

scaled_model = SVC(kernel="linear", C=100)
scaled_model.fit(X_scaled, y_scale)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))

DecisionBoundaryDisplay.from_estimator(
    unscaled_model,
    X_scale,
    response_method="decision_function",
    plot_method="contour",
    levels=[-1, 0, 1],
    linestyles=["--", "-", "--"],
    ax=axes[0]
)

axes[0].scatter(X_scale[:, 0], X_scale[:, 1], c=y_scale)
axes[0].set_title("Without scaling")
axes[0].set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")

DecisionBoundaryDisplay.from_estimator(
    scaled_model,
    X_scaled,
    response_method="decision_function",
    plot_method="contour",
    levels=[-1, 0, 1],
    linestyles=["--", "-", "--"],
    ax=axes[1]
)

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_scale)
axes[1].set_title("After standardization")
axes[1].set_xlabel("Standardized $x_1$")
axes[1].set_ylabel("Standardized $x_2$")

plt.tight_layout()
plt.show()

::: {.callout-warning}
## Scale the features

For SVMs, scaling should normally be part of the modeling pipeline. The scaler must be fitted only on the training data to prevent data leakage.
:::

A robust Scikit-Learn workflow is:

In [ ]:
#| label: linear-svc-pipeline

svm_clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1, loss="hinge", dual=True, random_state=42)
)

svm_clf.fit(X, y)

## Hard-margin classification

A hard-margin classifier requires every training observation to be correctly classified and to remain outside the margin:

$$
y_i\left(\mathbf{w}^{\mathsf T}\mathbf{x}_i+b\right) \geq 1,
\qquad i=1,\ldots,n.
$$

The optimization problem is

$$
\min_{\mathbf{w},b}
\frac{1}{2}\lVert\mathbf{w}\rVert^2
$$

subject to

$$
y_i\left(\mathbf{w}^{\mathsf T}\mathbf{x}_i+b\right) \geq 1.
$$

Hard-margin classification has two important limitations:

1. it works only when the data are linearly separable;
2. it is highly sensitive to outliers.

A single unusual observation may substantially alter the separating hyperplane or make perfect separation impossible.

## Soft-margin classification

Soft-margin classification allows some observations to fall inside the margin or even on the wrong side of the decision boundary. It introduces slack variables $\xi_i \geq 0$:

$$
y_i\left(\mathbf{w}^{\mathsf T}\mathbf{x}_i+b\right)
\geq 1-\xi_i.
$$

The corresponding objective is

$$
\min_{\mathbf{w},b,\boldsymbol{\xi}}
\frac{1}{2}\lVert\mathbf{w}\rVert^2
+
C\sum_{i=1}^{n}\xi_i.
$$

The hyperparameter $C$ controls the trade-off between a wide margin and margin violations:

- **large `C`** penalizes violations strongly and usually produces a narrower margin;
- **small `C`** tolerates more violations and usually produces a wider, more regularized margin.

### Visualizing the effect of `C`

In [ ]:
#| label: fig-svm-c
#| fig-cap: Effect of the regularization parameter C on the SVM margin.
#| code-fold: true
#| code-summary: Show code

iris = load_iris()
X_c = iris.data[:, (2, 3)]
y_c = (iris.target == 2).astype(int)

models = [
    make_pipeline(StandardScaler(), SVC(kernel="linear", C=0.1)),
    make_pipeline(StandardScaler(), SVC(kernel="linear", C=100))
]

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))

for model, ax, c_value in zip(models, axes, [0.1, 100]):
    model.fit(X_c, y_c)

    DecisionBoundaryDisplay.from_estimator(
        model,
        X_c,
        response_method="decision_function",
        plot_method="contour",
        levels=[-1, 0, 1],
        linestyles=["--", "-", "--"],
        ax=ax
    )

    ax.scatter(X_c[y_c == 0, 0], X_c[y_c == 0, 1], marker="o")
    ax.scatter(X_c[y_c == 1, 0], X_c[y_c == 1, 1], marker="s")
    ax.set_title(f"C = {c_value}")
    ax.set_xlabel("Petal length (cm)")
    ax.set_ylabel("Petal width (cm)")

plt.tight_layout()
plt.show()

A very large `C` may reduce training errors but create a model with high variance. A smaller `C` generally increases regularization and may improve generalization.

## Hinge loss

The hinge loss for observation $i$ is

$$
L_i =
\max\left(
0,\,
1-y_i f(\mathbf{x}_i)
\right),
$$

where

$$
f(\mathbf{x}_i)=\mathbf{w}^{\mathsf T}\mathbf{x}_i+b.
$$

The value

$$
y_i f(\mathbf{x}_i)
$$

is the signed functional margin.

Three cases are possible:

- if $y_i f(\mathbf{x}_i)\geq 1$, the observation is correctly classified and outside the margin, so the loss is zero;
- if $0<y_i f(\mathbf{x}_i)<1$, it is correctly classified but inside the margin;
- if $y_i f(\mathbf{x}_i)\leq 0$, it is misclassified.

In [ ]:
#| label: fig-hinge-loss
#| fig-cap: Hinge loss as a function of the signed margin.
#| code-fold: true
#| code-summary: Show code

signed_margin = np.linspace(-2, 3, 500)
hinge_loss = np.maximum(0, 1 - signed_margin)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(signed_margin, hinge_loss)
ax.axvline(0, linestyle="--")
ax.axvline(1, linestyle="--")
ax.set_xlabel(r"Signed margin $y f(\mathbf{x})$")
ax.set_ylabel("Hinge loss")
ax.set_title("Hinge loss")
plt.show()

::: {.callout-important}
## SVM scores are not probabilities

By default, SVM classifiers produce decision scores rather than class probabilities. `SVC(probability=True)` can estimate probabilities using an additional calibration procedure, but this increases training time.
:::

## Nonlinear SVM classification

A linear boundary is insufficient when the classes are not linearly separable in the original feature space. SVMs address this problem in two main ways:

1. explicitly create nonlinear features;
2. use a kernel that computes inner products in an implicit feature space.

### Polynomial features

A simple strategy is to transform the original inputs into polynomial features and then fit a linear SVM.

In [ ]:
#| label: polynomial-feature-svm

X_moons, y_moons = make_moons(
    n_samples=200,
    noise=0.15,
    random_state=42
)

polynomial_svm = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False),
    StandardScaler(),
    LinearSVC(C=10, loss="hinge", max_iter=20_000, random_state=42)
)

polynomial_svm.fit(X_moons, y_moons)

In [ ]:
#| label: fig-polynomial-features
#| fig-cap: Linear SVM fitted after a third-degree polynomial feature transformation.
#| code-fold: true

fig, ax = plt.subplots(figsize=(7, 5))

DecisionBoundaryDisplay.from_estimator(
    polynomial_svm,
    X_moons,
    response_method="decision_function",
    plot_method="contourf",
    alpha=0.25,
    ax=ax
)

ax.scatter(
    X_moons[y_moons == 0, 0],
    X_moons[y_moons == 0, 1],
    marker="o",
    label="Class 0"
)

ax.scatter(
    X_moons[y_moons == 1, 0],
    X_moons[y_moons == 1, 1],
    marker="s",
    label="Class 1"
)

ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend()
plt.show()

Explicit polynomial expansion is intuitive, but the number of generated features can increase quickly with the degree and the original number of predictors.

## The kernel trick

A kernel computes a similarity measure between two observations:

$$
K(\mathbf{x},\mathbf{x}')
=
\phi(\mathbf{x})^{\mathsf T}\phi(\mathbf{x}'),
$$

where $\phi(\cdot)$ represents a potentially high-dimensional feature transformation.

The kernel trick makes it possible to work with the inner products in that transformed space without explicitly computing all transformed features.

Common kernels include:

### Linear kernel

$$
K(\mathbf{x},\mathbf{x}')
=
\mathbf{x}^{\mathsf T}\mathbf{x}'.
$$

### Polynomial kernel

$$
K(\mathbf{x},\mathbf{x}')
=
\left(
\gamma\,\mathbf{x}^{\mathsf T}\mathbf{x}'
+r
\right)^d,
$$

where:

- $d$ is the polynomial degree;
- $\gamma$ controls the influence of the inner product;
- $r$, represented by `coef0` in Scikit-Learn, controls the contribution of lower- and higher-order terms.

### Radial basis function kernel

$$
K(\mathbf{x},\mathbf{x}')
=
\exp\left(
-\gamma
\lVert\mathbf{x}-\mathbf{x}'\rVert^2
\right).
$$

The RBF kernel is one of the most commonly used nonlinear kernels.

## Polynomial kernel

In [ ]:
#| label: polynomial-kernel-svm

poly_kernel_svm = make_pipeline(
    StandardScaler(),
    SVC(
        kernel="poly",
        degree=3,
        coef0=1,
        C=5
    )
)

poly_kernel_svm.fit(X_moons, y_moons)

In [ ]:
#| label: fig-polynomial-kernel
#| fig-cap: Decision boundary produced by a third-degree polynomial kernel.
#| code-fold: true

fig, ax = plt.subplots(figsize=(7, 5))

DecisionBoundaryDisplay.from_estimator(
    poly_kernel_svm,
    X_moons,
    response_method="decision_function",
    plot_method="contourf",
    alpha=0.25,
    ax=ax
)

ax.scatter(X_moons[y_moons == 0, 0], X_moons[y_moons == 0, 1], marker="o")
ax.scatter(X_moons[y_moons == 1, 0], X_moons[y_moons == 1, 1], marker="s")

ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
plt.show()

If the model underfits, possible adjustments include increasing `degree`, increasing `C`, or changing `coef0`. If it overfits, the reverse adjustments may help.

## Similarity features

A nonlinear transformation can also be constructed by measuring the similarity between an observation and selected landmarks.

Using a Gaussian radial basis function centered at landmark $\boldsymbol{\ell}$,

$$
\phi_{\boldsymbol{\ell}}(\mathbf{x})
=
\exp\left(
-\gamma
\lVert\mathbf{x}-\boldsymbol{\ell}\rVert^2
\right).
$$

A small distance from the landmark produces a value close to 1, whereas a large distance produces a value close to 0.

In [ ]:
#| label: fig-rbf-similarity
#| fig-cap: 'Gaussian RBF similarity features. Left: observations in the original one-dimensional space and their similarity to two landmarks. Right: observations in the transformed feature space.'
#| code-fold: true
#| code-summary: Show code

def gaussian_rbf(X, landmark, gamma):
    """
    Compute the Gaussian radial basis function similarity
    between each observation in X and a landmark.
    """
    return np.exp(
        -gamma * np.linalg.norm(X - landmark, axis=1) ** 2
    )


# One-dimensional observations
X_1d = np.linspace(-4, 4, 9).reshape(-1, 1)

# Binary class labels
y_rbf = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0])

# Landmarks and RBF parameter
landmark_1 = -2
landmark_2 = 1
gamma = 0.3

# Continuous grid used to draw the similarity functions
x_grid = np.linspace(-4.5, 4.5, 500).reshape(-1, 1)

similarity_landmark_1 = gaussian_rbf(
    x_grid,
    landmark_1,
    gamma
)

similarity_landmark_2 = gaussian_rbf(
    x_grid,
    landmark_2,
    gamma
)

# Transform the original observations into two similarity features
X_transformed = np.c_[
    gaussian_rbf(X_1d, landmark_1, gamma),
    gaussian_rbf(X_1d, landmark_2, gamma)
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(9.5, 4)
)

# ---------------------------------------------------------
# Left panel: original one-dimensional feature space
# ---------------------------------------------------------

ax = axes[0]

ax.axhline(
    y=0,
    linewidth=1
)

ax.scatter(
    [landmark_1, landmark_2],
    [0, 0],
    s=150,
    alpha=0.5,
    label="Landmarks"
)

ax.plot(
    X_1d[y_rbf == 0, 0],
    np.zeros(np.sum(y_rbf == 0)),
    "s",
    label="Class 0"
)

ax.plot(
    X_1d[y_rbf == 1, 0],
    np.zeros(np.sum(y_rbf == 1)),
    "^",
    label="Class 1"
)

ax.plot(
    x_grid[:, 0],
    similarity_landmark_1,
    "--",
    label=fr"Similarity to $\ell_1={landmark_1}$"
)

ax.plot(
    x_grid[:, 0],
    similarity_landmark_2,
    ":",
    label=fr"Similarity to $\ell_2={landmark_2}$"
)

ax.annotate(
    r"$\mathbf{x}$",
    xy=(X_1d[3, 0], 0),
    xytext=(-0.5, 0.22),
    ha="center",
    arrowprops={
        "arrowstyle": "->"
    },
    fontsize=14
)

ax.text(
    landmark_1,
    0.90,
    r"$\ell_1$",
    ha="center",
    fontsize=14
)

ax.text(
    landmark_2,
    0.90,
    r"$\ell_2$",
    ha="center",
    fontsize=14
)

ax.set_xlabel(r"Original feature $x_1$")
ax.set_ylabel("Similarity")
ax.set_xlim(-4.5, 4.5)
ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper right")


# ---------------------------------------------------------
# Right panel: transformed similarity-feature space
# ---------------------------------------------------------

ax = axes[1]

ax.axhline(
    y=0,
    linewidth=1
)

ax.axvline(
    x=0,
    linewidth=1
)

ax.plot(
    X_transformed[y_rbf == 0, 0],
    X_transformed[y_rbf == 0, 1],
    "s",
    label="Class 0"
)

ax.plot(
    X_transformed[y_rbf == 1, 0],
    X_transformed[y_rbf == 1, 1],
    "^",
    label="Class 1"
)

ax.annotate(
    r"$\phi(\mathbf{x})$",
    xy=(
        X_transformed[3, 0],
        X_transformed[3, 1]
    ),
    xytext=(0.68, 0.52),
    ha="center",
    arrowprops={
        "arrowstyle": "->"
    },
    fontsize=14
)

# Illustrative linear decision boundary
x_boundary = np.array([-0.1, 1.1])
y_boundary = 0.51 - 0.61 * x_boundary

ax.plot(
    x_boundary,
    y_boundary,
    "--",
    linewidth=2.5,
    label="Linear decision boundary"
)

ax.set_xlabel(
    r"$x_2=\phi_{\ell_1}(\mathbf{x})$"
)

ax.set_ylabel(
    r"$x_3=\phi_{\ell_2}(\mathbf{x})$"
)

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 1.1)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

The left panel shows the original one-dimensional observations and
their Gaussian RBF similarity to two landmarks located at
$\ell_1=-2$ and $\ell_2=1$.

Each original observation is transformed into two new features:

$$
x_2=\phi_{\ell_1}(\mathbf{x})=\exp\left(-\gamma\lVert \mathbf{x}-\ell_1\rVert^2\right),
$$

and

$$
x_3=\phi_{\ell_2}(\mathbf{x})=\exp\left(-\gamma\lVert \mathbf{x}-\ell_2\rVert^2\right).
$$

The right panel represents the observations in the transformed
feature space $(x_2,x_3)$. Although the classes are not linearly
separable in the original one-dimensional space, the RBF
transformation makes it possible to separate them using a linear
decision boundary.

Increasing `gamma` makes each similarity curve narrower. Decreasing it makes the curve wider.

## Gaussian RBF kernel

An RBF SVM can learn complex nonlinear boundaries without explicitly creating one feature per landmark.

In [ ]:
#| label: rbf-kernel-svm

rbf_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", gamma=5, C=1)
)

rbf_svm.fit(X_moons, y_moons)

### Interpreting `gamma`

The hyperparameter `gamma` controls how rapidly the similarity decreases with distance:

- **large `gamma`** gives each training observation a narrow region of influence and can produce an irregular boundary;
- **small `gamma`** gives observations a broader region of influence and produces a smoother boundary.

Thus, `gamma` behaves as an inverse measure of the radius of influence.

### Interaction between `C` and `gamma`

The following figure compares several combinations.

In [ ]:
#| label: fig-rbf-grid
#| fig-cap: Interaction between C and gamma for an RBF SVM.
#| code-fold: true
#| code-summary: Show code

parameter_combinations = [
    (0.1, 0.1),
    (100, 0.1),
    (0.1, 5),
    (100, 5)
]

fig, axes = plt.subplots(2, 2, figsize=(9.5, 8))

for ax, (c_value, gamma_value) in zip(
    axes.ravel(),
    parameter_combinations
):
    model = make_pipeline(
        StandardScaler(),
        SVC(
            kernel="rbf",
            C=c_value,
            gamma=gamma_value
        )
    )

    model.fit(X_moons, y_moons)

    DecisionBoundaryDisplay.from_estimator(
        model,
        X_moons,
        response_method="decision_function",
        plot_method="contourf",
        alpha=0.25,
        ax=ax
    )

    ax.scatter(
        X_moons[y_moons == 0, 0],
        X_moons[y_moons == 0, 1],
        marker="o"
    )

    ax.scatter(
        X_moons[y_moons == 1, 0],
        X_moons[y_moons == 1, 1],
        marker="s"
    )

    ax.set_title(
        fr"$C={c_value}$, $\gamma={gamma_value}$"
    )

    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")

plt.tight_layout()
plt.show()

A large `C` combined with a large `gamma` can fit highly localized patterns and may overfit. A small `C` combined with a small `gamma` produces a smoother and more regularized model.

::: {.callout-tip}
## Practical tuning strategy

For an RBF SVM:

1. standardize all numerical predictors;
2. begin with logarithmic grids for `C` and `gamma`;
3. evaluate combinations with cross-validation;
4. inspect both predictive performance and the gap between training and validation scores.
:::

## Multiclass classification

SVMs are fundamentally binary classifiers, but Scikit-Learn extends them to multiclass problems.

`SVC` uses a one-versus-one strategy internally. For $K$ classes, it trains

$$
\frac{K(K-1)}{2}
$$

binary classifiers.

`LinearSVC` uses a one-versus-rest strategy by default, fitting one classifier for each class.

In [ ]:
#| label: multiclass-svm

iris = load_iris()

multiclass_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=1, gamma="scale")
)

multiclass_svm.fit(iris.data, iris.target)

prediction = multiclass_svm.predict(
    iris.data[[0]]
)

prediction

## SVM regression

SVMs can also be used for regression. The objective is reversed relative to classification: the model attempts to place as many observations as possible inside an $\varepsilon$-wide tube around the regression function.

For prediction function

$$
f(\mathbf{x})
=
\mathbf{w}^{\mathsf T}\mathbf{x}+b,
$$

the $\varepsilon$-insensitive loss is

$$
L_{\varepsilon}
=
\max\left(
0,\,
|y-f(\mathbf{x})|-\varepsilon
\right).
$$

Errors smaller than $\varepsilon$ are ignored.

### Linear SVR

In [ ]:
#| label: fig-linear-svr
#| fig-cap: Linear support vector regression with two values of epsilon.
#| code-fold: true

rng = np.random.default_rng(42)

X_reg = 2 * rng.random((80, 1))
y_reg = 4 + 3 * X_reg[:, 0] + rng.normal(0, 0.8, 80)

epsilon_values = [0.2, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

x_plot = np.linspace(0, 2, 300).reshape(-1, 1)

for epsilon, ax in zip(epsilon_values, axes):
    model = make_pipeline(
        StandardScaler(),
        LinearSVR(
            epsilon=epsilon,
            C=1,
            dual="auto",
            max_iter=20_000,
            random_state=42
        )
    )

    model.fit(X_reg, y_reg)
    y_plot = model.predict(x_plot)

    ax.scatter(X_reg[:, 0], y_reg)
    ax.plot(x_plot[:, 0], y_plot)
    ax.plot(x_plot[:, 0], y_plot + epsilon, linestyle="--")
    ax.plot(x_plot[:, 0], y_plot - epsilon, linestyle="--")
    ax.set_title(fr"$\varepsilon={epsilon}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")

plt.tight_layout()
plt.show()

The model is called $\varepsilon$-insensitive because movements of training observations within the tube do not affect the loss.

### Nonlinear SVR

A kernelized SVR can model nonlinear relationships.

In [ ]:
#| label: fig-polynomial-svr
#| fig-cap: Polynomial-kernel SVR with different values of C.
#| code-fold: true

rng = np.random.default_rng(42)

X_quad = 2 * rng.random((100, 1)) - 1
y_quad = (
    0.2
    + 0.1 * X_quad[:, 0]
    + 0.5 * X_quad[:, 0] ** 2
    + rng.normal(0, 0.08, 100)
)

models = [
    SVR(kernel="poly", degree=2, C=100, epsilon=0.1, coef0=1),
    SVR(kernel="poly", degree=2, C=0.01, epsilon=0.1, coef0=1)
]

x_plot = np.linspace(-1, 1, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for model, ax, c_value in zip(models, axes, [100, 0.01]):
    model.fit(X_quad, y_quad)
    y_plot = model.predict(x_plot)

    ax.scatter(X_quad[:, 0], y_quad)
    ax.plot(x_plot[:, 0], y_plot)
    ax.plot(x_plot[:, 0], y_plot + model.epsilon, linestyle="--")
    ax.plot(x_plot[:, 0], y_plot - model.epsilon, linestyle="--")
    ax.set_title(fr"Polynomial SVR, $C={c_value}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")

plt.tight_layout()
plt.show()

A larger `C` reduces regularization and attempts to fit the training observations more closely. A smaller `C` produces a smoother model.

## Computational considerations

The main Scikit-Learn SVM estimators have different computational characteristics.

| Estimator | Main use | Kernel support | Typical behavior |
|---|---|---:|---|
| `LinearSVC` | Linear classification | No | Scales comparatively well to large datasets |
| `SVC(kernel="linear")` | Linear classification | Yes | Useful when support vectors are required |
| `SVC(kernel="rbf")` | Nonlinear classification | Yes | Powerful but expensive for large datasets |
| `LinearSVR` | Linear regression | No | Suitable for larger datasets |
| `SVR` | Nonlinear regression | Yes | Can become slow as sample size increases |

Kernel SVMs usually require substantial computation as the number of observations grows. Their training time can increase rapidly because the algorithm works with pairwise relationships between observations.

## Practical modeling workflow

A sound SVM workflow includes:

1. split the data into training and test sets;
2. place preprocessing and SVM estimation in one pipeline;
3. standardize numerical variables;
4. select a kernel according to the expected boundary complexity;
5. tune `C`, and for RBF models tune `gamma`;
6. use cross-validation for model selection;
7. evaluate the final model once on untouched test data;
8. inspect class imbalance and choose suitable metrics.

Example pipeline:

In [ ]:
#| label: practical-svm-pipeline

svm_pipeline = make_pipeline(
    StandardScaler(),
    SVC(
        kernel="rbf",
        C=1,
        gamma="scale"
    )
)

## Common mistakes

::: {.callout-warning}
## Data leakage

Do not standardize the complete dataset before splitting it. Fit the scaler only through a pipeline trained on the training data.
:::

::: {.callout-warning}
## Unscaled predictors

Using predictors with very different scales may distort the geometry of the problem and produce a poor decision boundary.
:::

::: {.callout-warning}
## Interpreting `C` incorrectly

In Scikit-Learn, a larger `C` means **less regularization**, while a smaller `C` means **more regularization**.
:::

::: {.callout-warning}
## Tuning on the test set

The test set should not be used to select `C`, `gamma`, the kernel, or any preprocessing decision.
:::

## Chapter summary

Support Vector Machines build decision boundaries by maximizing the margin between classes. The closest training observations, called support vectors, determine the boundary. Hard-margin classification requires perfect linear separation, while soft-margin classification allows violations controlled by `C`.

Nonlinear SVMs use kernels to represent complex boundaries without explicitly generating all transformed features. The polynomial and RBF kernels are common choices. In an RBF model, `gamma` controls the radius of influence of each observation, while `C` controls the penalty for margin violations.

SVM regression uses an $\varepsilon$-insensitive tube and ignores prediction errors smaller than $\varepsilon$. In both classification and regression, feature scaling and careful hyperparameter tuning are essential.

## Exercises

1. Fit linear SVM classifiers to the Iris dataset using several values of `C`. Compare the number of support vectors and the width of the margin.

2. Generate a moons dataset with a higher noise level. Compare a polynomial-feature SVM, a polynomial-kernel SVM, and an RBF SVM.

3. For an RBF classifier, create a logarithmic grid of values for `C` and `gamma`. Use cross-validation to select the best combination.

4. Explain why increasing `gamma` may lead to overfitting.

5. Compare `LinearSVC` and `SVC(kernel="linear")` on the same standardized dataset. Examine their accuracy, training time, and available attributes.

6. Fit an `SVR` model to a nonlinear regression dataset. Study how the fitted curve changes when `C`, `epsilon`, and `gamma` are varied.